# SCALE x ODYSSEY -- 02: Training

Interactive training with live plots.

In [ ]:
import sys
sys.path.insert(0, '../src')

from utils import load_config, set_seed
from dataset import get_loaders
from model import AstroClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

config = load_config('../configs/config.yaml')
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load Data

In [ ]:
train_loader, val_loader, test_loader = get_loaders(
    config['data']['processed_dir'],
    batch_size=config['training']['batch_size'],
    num_workers=config['data']['num_workers'],
    image_size=config['data']['image_size'])

## Initialize Model

In [ ]:
model = AstroClassifier(
    num_classes=config['model']['num_classes'],
    backbone=config['model']['backbone'],
    pretrained=True,
    dropout=config['model']['dropout']).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=config['training']['label_smoothing'])
optimizer = optim.AdamW(model.parameters(), lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])
scheduler = OneCycleLR(optimizer, max_lr=config['training']['lr'], epochs=config['training']['num_epochs'], steps_per_epoch=len(train_loader))
scaler = GradScaler()
writer = SummaryWriter(log_dir=config['paths']['logs'])

train_losses, val_accs = [], []
for epoch in range(config['training']['num_epochs']):
    model.train()
    epoch_loss = 0
    for images, labels in tqdm(train_loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        epoch_loss += loss.item() * images.size(0)
    train_loss = epoch_loss / len(train_loader.dataset)
    train_losses.append(train_loss)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    val_accs.append(val_acc)
    writer.add_scalars('Loss', {'train': train_loss}, epoch)
    writer.add_scalar('Accuracy/val', val_acc, epoch)
    print(f'Epoch {epoch+1:02d} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}')
writer.close()

## Training Curves

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(train_losses, 'b-', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='b')
ax2 = ax1.twinx()
ax2.plot(val_accs, 'r-', label='Val Accuracy')
ax2.set_ylabel('Accuracy', color='r')
ax2.set_ylim(0, 1)
fig.legend(loc='upper right')
plt.title('Training Progress')
plt.tight_layout()
plt.show()